###Understanding the data

In [40]:
#importing necessary packages
import pandas as pd
from sklearn.datasets import fetch_openml

In [41]:
#load data from csv
freq = fetch_openml(data_id=41214, as_frame=True).data
sev = fetch_openml(data_id=41215, as_frame=True).data

print(freq.info())
print(sev.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 678013 entries, 0 to 678012
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   IDpol       678013 non-null  float64 
 1   ClaimNb     678013 non-null  int64   
 2   Exposure    678013 non-null  float64 
 3   Area        678013 non-null  category
 4   VehPower    678013 non-null  int64   
 5   VehAge      678013 non-null  int64   
 6   DrivAge     678013 non-null  int64   
 7   BonusMalus  678013 non-null  int64   
 8   VehBrand    678013 non-null  category
 9   VehGas      678013 non-null  object  
 10  Density     678013 non-null  int64   
 11  Region      678013 non-null  category
dtypes: category(3), float64(2), int64(6), object(1)
memory usage: 48.5+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26639 entries, 0 to 26638
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   IDpol    

The *freq* dataset contains 678,013 entries with 12 features including policy ID *IDpol* and frequency *ClaimNb*.
The *sev* dataset contains 26,639 entries with 2 features *IDpol* and Severity *ClaimAmount*.
We note that the number of entries of the two datasets differs vastly. This however is to be expected, since most policies will not have a claim.
Before merging the dataframes, we check for potential duplicates.

In [63]:
print(f'{freq.duplicated(subset='IDpol').sum()} duplicates in IDpol column of freq DataFrame')
print(f'{sev.duplicated(subset='IDpol').sum()} duplicates in IDpol column of sev DataFrame')
print(f'{(freq['ClaimNb']>1).sum()} IDs with multiple claims in freq dataset')<

0 duplicates in IDpol column of freq DataFrame
1689 duplicates in IDpol column of sev DataFrame
1882 IDs with multiple claims in freq dataset


Considering the policy ID, there are 0 duplicates in the frequency dataset. However, there are 1689 duplicates in the severity dataset, which is most likely due to multiple claims per policy. Still, this is not a one-to-one relation as there are more IDs with multiple claims in the frequency data than duplicates in the severity data.
Before moving on we aggregate the sev dataframe, resulting in unique identifiers.

In [43]:
#sum ClaimAmount over IDs
sev_agg = sev.groupby('IDpol').agg(ClaimAmount=('ClaimAmount','sum')).reset_index()

In [64]:
print(f'Number of IDs included in sev data but no in freq: {sev_agg[~sev_agg['IDpol'].isin(freq['IDpol'])].shape[0]}')

Number of IDs included in sev data but no in freq: 6


There are 6 rows in the sev data not included in the freq data. We will drop these using a left-join.

In [45]:
#merge data into one df, using left-join, so only IDs having features are used
df = pd.merge(freq,sev_agg,on='IDpol',how='left',validate='one_to_one')

#fill rows with no ClaimAmount with 0
df['ClaimAmount'] = df['ClaimAmount'].fillna(0)

The dataframes where merged successfully with a one-to-one relation.